# **Pertemuan 3 - PREPROCESSING 2**
### **STEMMING And Regular Expression (RegEx)**


## **📖 Pengantar Materi**

**🔧 STEMMING**

Stemming itu adalah proses mengubah kata berimbuhan (seperti "berlari", "pelari", "berlarian") menjadi bentuk dasarnya ("lari"). Ini sering dipakai dalam bidang Natural Language Processing (NLP) untuk analisis teks, supaya kata-kata serupa dianggap sama.

Contoh stemming bahasa Indonesia:

bermain → main

berlari → lari

pelari → lari

Kalau di bahasa Inggris:

running → run

studies → study

playing → play

Biasanya stemming dilakukan menggunakan algoritma tertentu, seperti:

Porter Stemmer (untuk Bahasa Inggris)

Sastrawi (untuk Bahasa Indonesia)


**🧮 REGULAR EXPRESSION**

Regular expression (regex) adalah pola yang digunakan untuk mencocokkan karakter dalam string. Regex sering digunakan dalam NLP untuk membersihkan atau mengekstrak bagian tertentu dari teks. RegEx digunakan untuk menemukan pola dalam teks, membersihkan data, dan validasi input.

**Contoh Pola RegEx Umum:**

Hapus angka
```
#[\d+] (Mencari satu atau lebih digit)
no_numbers = re.sub(r'\d+', '', text)
```
Hapus simbol
```
#[^a-zA-Z\s] (Menyisakan huruf & spasi)
```
Cari email
```
#[\S+@\S+] (Format dasar email)
emails = re.findall(r'\S+@\S+', text)
```
Ganti spasi ganda
```
#[\s+] (Satu atau lebih spasi)
```

**💡Tentang Dataset yang akan digunakan**

Dataset adalah kumpulan data yang akan diproses. Biasanya berbentuk file CSV, kadangkala excel, pdf, dll. Struktur utama dataset adalah **baris (row/index) dan kolom (column/labels/features).** Dataset untuk PBA bisa didapatkan dari mana saja, misal: review Google Maps tentang suatu tempat, kumpulan artikel berita, komentar di medsos (YT/IG/X), dll.

Pada pertemuan ini, kita akan menggunakan dataset kumpulan tweet dengan kata kunci "efisiensi" yang di-crawling dari X/Twitter per tanggal 9-13 Maret 2025.


**🎯Goals setelah mengikuti praktikum hari ini:**
*   Bisa menerapkan cleaning dan tokenize menggunakan Regular Expression




## **🪄 Langkah Praktikum**

## **A. Preparation**

Checklist:
- Import library
- Mempersiapkan dataset

###**1. Import Library**

In [2]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.7 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import nltk
nltk.download('punkt')
nltk.download('stopwords')

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
stopwords_ind = stopwords.words('indonesian')

import re  # Import the 're' module for regular expressions
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory # Import Sastrawi for stemming

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### **2. Download Dataset**

Sudah otomatis terdownload. Untuk memastikan silakan dicek dengan klik ikon folder di sidebar kiri.

In [6]:
!wget https://raw.githubusercontent.com/jiorjioo/PBA2025/refs/heads/main/tweet_efisiensi.csv

--2026-10-20 10:15:36--  https://raw.githubusercontent.com/jiorjioo/PBA2025/refs/heads/main/tweet_efisiensi.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 865314 (845K) [text/plain]
Saving to: ‘tweet_efisiensi.csv’

tweet_efisiensi.csv 100%[===================>] 845.03K  --.-KB/s    in 0.05s   

2026-10-20 10:15:37 (16.5 MB/s) - ‘tweet_efisiensi.csv’ saved [865314/865314]



### **3. Load/read dataset**

 Menggunakan fungsi read_csv dari library Pandas

In [7]:
# df adalah variabel yang akan menampung dataset agar mudah dipanggil
df = pd.read_csv('tweet_efisiensi.csv')
df

# klik ikon tabel (kotak-kotak) berwarna biru di paling kanan agar bisa melihat tampilan data yang lebih rapi

,conversation_id_str,created_at,favorite_count,full_text,id_str,image_url,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,tweet_url,user_id_str,username
0,1900328982928908377,Thu Mar 13 23:31:50 +0000 2025,0,Danantara Indonesia berkomitmen untuk mendoron...,1900328982928908377,https://pbs.twimg.com/media/Gl9SzEzWkAAv9cx.jpg,NaN,in,Papua Barat,0,1,0,https://x.com/NabilaKeisha7/status/19003289829...,1358952377609408513,NabilaKeisha7
1,1900150673057972397,Thu Mar 13 23:31:20 +0000 2025,0,@Mdy_Asmara1701 Kalau bs efisiensi Logikanya a...,1900328856718049776,NaN,Mdy_Asmara1701,in,NaN,0,0,0,https://x.com/suharto91327871/status/190032885...,1260193672248737795,suharto91327871
2,1900064433990492302,Thu Mar 13 23:29:40 +0000 2025,0,@CNNIndonesia Apa sih buk! Noh efisiensi tapi ...,1900328436117422091,NaN,CNNIndonesia,in,NaN,0,0,0,https://x.com/bluish_66/status/190032843611742...,1595788799849553920,bluish_66
3,1900150673057972397,Thu Mar 13 23:27:17 +0000 2025,0,@Mdy_Asmara1701 Efisiensi cuma lari ke dananta...,1900327836311052770,NaN,Mdy_Asmara1701,in,NaN,0,0,0,https://x.com/wadestahh/status/190032783631105...,1880516316920266752,wadestahh
4,1900111771035423005,Thu Mar 13 23:22:25 +0000 2025,0,@geloraco Kiw kiwww hiburan baru buat dana bar...,1900326614313165178,NaN,geloraco,in,NaN,0,0,0,https://x.com/bluish_66/status/190032661431316...,1595788799849553920,bluish_66
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2510,1898294500881277317,Sun Mar 09 05:21:25 +0000 2025,1,@islah_bahrawi Maju PSI!!! Hidup Efisiensi!! H...,1898605018003243031,NaN,islah_bahrawi,in,Indonesia,0,0,0,https://x.com/ariefrasyad/status/1898605018003...,38112151,ariefrasyad
2511,1898604885517762831,Sun Mar 09 05:20:53 +0000 2025,0,Indonesia jam 10 lampu harus udah mati semua g...,1898604885517762831,NaN,NaN,in,Shattering skies above,0,0,0,https://x.com/PrayForPredator/status/189860488...,315040271,PrayForPredator
2512,1898604583192244726,Sun Mar 09 05:19:41 +0000 2025,0,Kebenaran Efektif adalah hal yang tercermin da...,1898604583192244726,https://pbs.twimg.com/media/Glkxtz4aUAECa6m.jpg,NaN,in,Jahannam Lt. Underground,0,1,0,https://x.com/adencompany/status/1898604583192...,2605985815,adencompany
2513,1898462030094578132,Sun Mar 09 05:18:33 +0000 2025,0,@EricRicaldo @prabowo Presidennya Paham efisie...,1898604297690210536,NaN,EricRicaldo,in,NaN,0,0,0,https://x.com/HambaPembeda/status/189860429769...,208000238,HambaPembeda


## **B. Preprocessing**




Karena preprocessing pada PBA adalah proses yang berlanjut dan satu rangkaian, maka pertemuan kali ini merupakan lanjutan dari materi minggu lalu. Kita akan melakukan preprocessing antara lain:
- Data Cleaning menggunakan RegEx
- Stemming
- Tokenisasi

Poin yang diberi tanda "⭐" adalah fokus pada praktikum pertemuan 3 ini

In [9]:
df2 = df.head(200)
df2

,conversation_id_str,created_at,favorite_count,full_text,id_str,image_url,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,tweet_url,user_id_str,username
0,1900328982928908377,Thu Mar 13 23:31:50 +0000 2025,0,Danantara Indonesia berkomitmen untuk mendoron...,1900328982928908377,https://pbs.twimg.com/media/Gl9SzEzWkAAv9cx.jpg,NaN,in,Papua Barat,0,1,0,https://x.com/NabilaKeisha7/status/19003289829...,1358952377609408513,NabilaKeisha7
1,1900150673057972397,Thu Mar 13 23:31:20 +0000 2025,0,@Mdy_Asmara1701 Kalau bs efisiensi Logikanya a...,1900328856718049776,NaN,Mdy_Asmara1701,in,NaN,0,0,0,https://x.com/suharto91327871/status/190032885...,1260193672248737795,suharto91327871
2,1900064433990492302,Thu Mar 13 23:29:40 +0000 2025,0,@CNNIndonesia Apa sih buk! Noh efisiensi tapi ...,1900328436117422091,NaN,CNNIndonesia,in,NaN,0,0,0,https://x.com/bluish_66/status/190032843611742...,1595788799849553920,bluish_66
3,1900150673057972397,Thu Mar 13 23:27:17 +0000 2025,0,@Mdy_Asmara1701 Efisiensi cuma lari ke dananta...,1900327836311052770,NaN,Mdy_Asmara1701,in,NaN,0,0,0,https://x.com/wadestahh/status/190032783631105...,1880516316920266752,wadestahh
4,1900111771035423005,Thu Mar 13 23:22:25 +0000 2025,0,@geloraco Kiw kiwww hiburan baru buat dana bar...,1900326614313165178,NaN,geloraco,in,NaN,0,0,0,https://x.com/bluish_66/status/190032661431316...,1595788799849553920,bluish_66
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,1900124610269425813,Thu Mar 13 13:51:50 +0000 2025,0,@PrakosoGrego @Aerio_Ina Efisiensi tapi efekny...,1900183019010613305,NaN,PrakosoGrego,in,Jakarta,0,0,0,https://x.com/rodagenik/status/190018301901061...,1194220651155873793,rodagenik
196,1900041052154937833,Thu Mar 13 13:51:34 +0000 2025,0,@GunRomli Keren ih efisiensi segala bidang,1900182952740610131,NaN,GunRomli,in,"Banjarnegara, Indonesia",0,0,0,https://x.com/arifskellington/status/190018295...,2409647858,arifskellington
197,1900182002625896899,Thu Mar 13 13:51:32 +0000 2025,0,@officialinews_ Aparat penegak hukum tersangka...,1900182947350929871,NaN,officialinews_,in,Indonesia,0,0,0,https://x.com/deepsecretmagic/status/190018294...,1571595511857315842,deepsecretmagic
198,1900172500144972029,Thu Mar 13 13:50:44 +0000 2025,0,@tempodotco Efisiensi asu pak eh bagus sahkan ...,1900182745168699813,NaN,tempodotco,in,pontianak,0,0,0,https://x.com/ediaoe1/status/1900182745168699813,753077539,ediaoe1


In [10]:
#Slice 200 data teratas
df2 = df[:200]
df2

,conversation_id_str,created_at,favorite_count,full_text,id_str,image_url,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,tweet_url,user_id_str,username
0,1900328982928908377,Thu Mar 13 23:31:50 +0000 2025,0,Danantara Indonesia berkomitmen untuk mendoron...,1900328982928908377,https://pbs.twimg.com/media/Gl9SzEzWkAAv9cx.jpg,NaN,in,Papua Barat,0,1,0,https://x.com/NabilaKeisha7/status/19003289829...,1358952377609408513,NabilaKeisha7
1,1900150673057972397,Thu Mar 13 23:31:20 +0000 2025,0,@Mdy_Asmara1701 Kalau bs efisiensi Logikanya a...,1900328856718049776,NaN,Mdy_Asmara1701,in,NaN,0,0,0,https://x.com/suharto91327871/status/190032885...,1260193672248737795,suharto91327871
2,1900064433990492302,Thu Mar 13 23:29:40 +0000 2025,0,@CNNIndonesia Apa sih buk! Noh efisiensi tapi ...,1900328436117422091,NaN,CNNIndonesia,in,NaN,0,0,0,https://x.com/bluish_66/status/190032843611742...,1595788799849553920,bluish_66
3,1900150673057972397,Thu Mar 13 23:27:17 +0000 2025,0,@Mdy_Asmara1701 Efisiensi cuma lari ke dananta...,1900327836311052770,NaN,Mdy_Asmara1701,in,NaN,0,0,0,https://x.com/wadestahh/status/190032783631105...,1880516316920266752,wadestahh
4,1900111771035423005,Thu Mar 13 23:22:25 +0000 2025,0,@geloraco Kiw kiwww hiburan baru buat dana bar...,1900326614313165178,NaN,geloraco,in,NaN,0,0,0,https://x.com/bluish_66/status/190032661431316...,1595788799849553920,bluish_66
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,1900124610269425813,Thu Mar 13 13:51:50 +0000 2025,0,@PrakosoGrego @Aerio_Ina Efisiensi tapi efekny...,1900183019010613305,NaN,PrakosoGrego,in,Jakarta,0,0,0,https://x.com/rodagenik/status/190018301901061...,1194220651155873793,rodagenik
196,1900041052154937833,Thu Mar 13 13:51:34 +0000 2025,0,@GunRomli Keren ih efisiensi segala bidang,1900182952740610131,NaN,GunRomli,in,"Banjarnegara, Indonesia",0,0,0,https://x.com/arifskellington/status/190018295...,2409647858,arifskellington
197,1900182002625896899,Thu Mar 13 13:51:32 +0000 2025,0,@officialinews_ Aparat penegak hukum tersangka...,1900182947350929871,NaN,officialinews_,in,Indonesia,0,0,0,https://x.com/deepsecretmagic/status/190018294...,1571595511857315842,deepsecretmagic
198,1900172500144972029,Thu Mar 13 13:50:44 +0000 2025,0,@tempodotco Efisiensi asu pak eh bagus sahkan ...,1900182745168699813,NaN,tempodotco,in,pontianak,0,0,0,https://x.com/ediaoe1/status/1900182745168699813,753077539,ediaoe1


In [11]:
# Melakukan filtering dari 200 data yang telah dislice sebelumnya
# Filter data berdasarkan suatu kondisi (contoh: tweet yang mengandung kata “korup”) >> kalian bisa eksperimen juga mau pake kata apa
filter_df = df2[df2['full_text'].str.contains("korup", case=False, na=False)]
filter_df

,conversation_id_str,created_at,favorite_count,full_text,id_str,image_url,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,tweet_url,user_id_str,username
6,1900061823572828513,Thu Mar 13 23:19:04 +0000 2025,1,@tanyakanrl 2025: Coretax Danantara Mega korup...,1900325767789371826,NaN,tanyakanrl,in,NaN,0,0,0,https://x.com/MorningDew090/status/19003257677...,1388196981458104320,MorningDew090
8,1900325546699219128,Thu Mar 13 23:18:11 +0000 2025,0,Itu bukan solusi. Solusi &amp; efek jera untuk...,1900325546699219128,NaN,NaN,in,NaN,0,0,0,https://x.com/soetanahasby/status/190032554669...,210012158,soetanahasby
11,1900002189848043836,Thu Mar 13 23:14:43 +0000 2025,0,@mas_veel disesatkan pala lu bau menyan...daya...,1900324673625481680,NaN,mas_veel,in,Dipangkuan Ibu Pertiwi,0,0,0,https://x.com/tiledjago/status/190032467362548...,149346411,tiledjago
32,1900198534575788253,Thu Mar 13 22:18:44 +0000 2025,1,@Dahnilanzar Efisiensi tapi bikin proyek baru ...,1900310585059008961,NaN,Dahnilanzar,in,nomaden,0,0,0,https://x.com/psyche_msa/status/19003105850590...,1567351806,psyche_msa
37,1900041052154937833,Thu Mar 13 22:12:11 +0000 2025,0,@GunRomli 32T itu kecil utk kelas APBN sedangk...,1900308936865239542,NaN,GunRomli,in,Indonesia - USA,0,0,0,https://x.com/maya_khatiri/status/190030893686...,1869394678594068480,maya_khatiri
42,1900305117708218604,Thu Mar 13 22:06:18 +0000 2025,0,Berjalannya waktu setidaknya ada dua wacana be...,1900307455852331155,NaN,whendarkness_,in,NaN,0,1,0,https://x.com/whendarkness_/status/19003074558...,1455248025492725766,whendarkness_
50,1900214422519599612,Thu Mar 13 21:30:26 +0000 2025,0,@kompascom Selama lingkaranya ga ada yg di ber...,1900298431459102925,NaN,kompascom,in,NaN,0,0,0,https://x.com/adheefx/status/1900298431459102925,1701537149622124544,adheefx
53,1900002189848043836,Thu Mar 13 21:28:24 +0000 2025,0,@mas_veel Dah nambah kali mungkin selama ini s...,1900297919707938975,NaN,mas_veel,in,"Lampung, Indonesia",0,0,0,https://x.com/JOINLampung/status/1900297919707...,949525285828833280,JOINLampung
64,1900294191080718826,Thu Mar 13 21:13:35 +0000 2025,17,Yg terhormat pak @prabowo. Klo mau berantas Ko...,1900294191080718826,NaN,NaN,in,"Jakarta Barat, DKI Jakarta",0,9,7,https://x.com/Bara__Akbar/status/1900294191080...,1613404169997783040,Bara__Akbar
70,1900292375005847764,Thu Mar 13 21:06:22 +0000 2025,0,Dunia sedang tidak baik SAJA termasuk kemungki...,1900292375005847764,https://pbs.twimg.com/media/Gl8xywLbgAAeTj9.jpg,NaN,in,NaN,0,0,1,https://x.com/AcoyBk/status/1900292375005847764,1359071931153018881,AcoyBk


In [12]:
# Extract the 'full_text' column
full_text_column = filter_df['full_text']

# Now you have the 'full_text' column in the 'full_text_column' variable.
# You can further process it or print it. For example, to print all data:
print(full_text_column)

6      @tanyakanrl 2025: Coretax Danantara Mega korup...
8      Itu bukan solusi. Solusi &amp; efek jera untuk...
11     @mas_veel disesatkan pala lu bau menyan...daya...
32     @Dahnilanzar Efisiensi tapi bikin proyek baru ...
37     @GunRomli 32T itu kecil utk kelas APBN sedangk...
42     Berjalannya waktu setidaknya ada dua wacana be...
50     @kompascom Selama lingkaranya ga ada yg di ber...
53     @mas_veel Dah nambah kali mungkin selama ini s...
64     Yg terhormat pak @prabowo. Klo mau berantas Ko...
70     Dunia sedang tidak baik SAJA termasuk kemungki...
110    @Dahnilanzar Dapat Nasihat siapa sih ini prabo...
111    @e100ss Ingat Pak lagi efisiensi. Dripada bang...
122    @h3ru_h Katanya efisiensi? Lha kog program ker...
133    @Dahnilanzar Padahal mendingan bikin TEMPAT PE...
138    @tanyakanrl perluasan lahan sawit disinformasi...
171    @Minnie_imut Sekolah yg udah ada bukan di manf...
191    Lah malah buat program baru bangun penjara khu...
198    @tempodotco Efisiensi as

### **1. Filtering**

#### 1a. Menghapus karakter spesial menggunakan Regular expression (RegEx) ⭐

In [13]:
# prompt: Lakukan cleaning terlebih dahulu

import re

# Membuat function untuk cleaning data
def clean_text(text):
  text = re.sub(r'http\S+', '', text)         # Remove URLs
  text = re.sub(r'@\S+', '', text)            # Remove mentions (@username)
  text = re.sub(r'#\S+', '', text)            # Remove hashtags (#hashtag)
  text = re.sub(r'[^a-zA-Z\s]', '', text)     # Remove special characters and numbers
  text = re.sub(r'\s+', ' ', text).strip()    # Remove extra whitespaces
  return text

In [14]:
# contoh penggunaan
raw_sample = filter_df['full_text'].iloc[0]
cleantext = clean_text(raw_sample)

# lihat bedanya antara isi variabel raw_sample dan cleantext
print('Raw data\t: ', raw_sample)
print('Cleaning\t: ', cleantext)

Raw data	:  @tanyakanrl 2025: Coretax Danantara Mega korupsi(PLN Pertamina BJB) KaburAjaDulu IndonesiaGelap Grooming &amp; Pedofil CPNS Dwifungsi Abri PHK Masal MBG Efisiensi Anggaran Banjir dll Satt baru jg maret dah banyak amat masalahnya
Cleaning	:  Coretax Danantara Mega korupsiPLN Pertamina BJB KaburAjaDulu IndonesiaGelap Grooming amp Pedofil CPNS Dwifungsi Abri PHK Masal MBG Efisiensi Anggaran Banjir dll Satt baru jg maret dah banyak amat masalahnya


#### 1b. TAMBAHAN: Contoh Regex untuk Pola Data yang Lain ⭐





 Penjelasan Simbol

 \d  : penanda angka di reguler Expression

 \s  : spasi

 "|" : adalah "atau"

 "?" : menyatakan pilihan (optional)

 compile : meng-compile pola untuk membuat objek regex baru

**POLA EMAIL**

In [15]:
# Inisialisasi passage/kalimat yang berisi e-mail
kalimat = "From: gantipakeemailmu@gmail.comTo: gantipakeemailmu2@webmail.uad.ac.id, gantipakeemailmu3@gmail.coSubject: Meeting at 2 PM Body:Hello,Just a reminder about our meeting today at 2 PM.Please confirm your availability.Thank you!"

# Inisialisasi pola regex untuk e-mail
email_pattern = r'[\w\.-]+@[\w\.-]+\w+'

In [17]:
# Mengambil e-mail pakai findall
ekstraksi = re.findall(email_pattern, kalimat)
ekstraksi

['gantipakeemailmu@gmail.comTo',
 'gantipakeemailmu2@webmail.uad.ac.id',
 'gantipakeemailmu3@gmail.coSubject']

In [ ]:
# Menghapus Email dalam teks
print( re.sub(email_pattern, '', kalimat))

From: : , : Meeting at 2 PM Body:Hello,Just a reminder about our meeting today at 2 PM.Please confirm your availability.Thank you!


**POLA NOMOR TELPON**

In [18]:
kalimat = 'Contact kami di 021-7632456 atau 021-443-4162 atau 021 797 4524 atau 0822959020 atau +628199258643'

phone_pattern = re.compile(r'(\d{3}[-\.\s]??\d{3}[-\.\s]??\d{4,6}|\(\d{3}\)\s*\d{3}[-\.\s]??\d{4,6}|\d{3,4}[-\.\s]??\d{4,6})')

In [20]:
# Sensor *** untuk pola Nomor Telepon
print(re.sub(phone_pattern, '***', kalimat))

Contact kami di *** atau *** atau *** atau *** atau +***


In [21]:
# Mengambil nomor telepon dari teks berdasarkan pola regex yang telah ditentukan
phone_numbers = re.findall(phone_pattern, kalimat)
print('Nomer telephone yang ditemukan: ', phone_numbers)

Nomer telephone yang ditemukan:  ['021-7632456', '021-443-4162', '021 797 4524', '0822959020', '628199258643']


**POLA URL**

In [24]:
kalimat = 'Teman teman dapat mengunjungi link berikut untuk sumber belajar: https://www.regular-expressions.info/ atau https://www.towardsdatascience.com atau https://wikipedia.org'
kalimat

'Teman teman dapat mengunjungi link berikut untuk sumber belajar: https://www.regular-expressions.info/ atau https://www.towardsdatascience.com atau https://wikipedia.org'

In [25]:
url_pattern = r'www|http:|https:+[^\s]+[\w/]'

# Menghapus url pada teks. Actually bukan ngehapus, melainkan ✨mengganti (substitute)✨ kata yang terseleksi regex dari keseluruhan kalimat dengan spasi
print(re.sub(url_pattern,' ', kalimat))

# Mengambil url dari teks berdasarkan pola regex yang telah ditentukan
urls = re.findall(url_pattern, kalimat)# get urls
print('URL yang ditemukan: ',urls)


Teman teman dapat mengunjungi link berikut untuk sumber belajar:   atau   atau  
URL yang ditemukan:  ['https://www.regular-expressions.info/', 'https://www.towardsdatascience.com', 'https://wikipedia.org']


**POLA PLAT MOTOR**

In [26]:
kalimat = "Mr.X punya 2 motor dan 1 mobil. Adapun plat nomornya yaitu B 234 AA, C 3244 YGS, dan D 3 S."
kalimat

'Mr.X punya 2 motor dan 1 mobil. Adapun plat nomornya yaitu B 234 AA, C 3244 YGS, dan D 3 S.'

In [27]:
plat_pattern = r'[A-Z]{1,2}\s\d{1,4}\s[A-Z]{1,3}'

# Menghapus plat kendaraan pada teks
print(re.sub(plat_pattern, ' ', kalimat))

Mr.X punya 2 motor dan 1 mobil. Adapun plat nomornya yaitu  ,  , dan  .


In [28]:
kumpulan_plat_kendaraan = re.findall(plat_pattern, kalimat)
print(kumpulan_plat_kendaraan)

['B 234 AA', 'C 3244 YGS', 'D 3 S']


#### 1c. Casefolding

In [29]:
def casefolding(text):
  text = text.lower()                               # Mengubah teks menjadi lower case
  return text

In [30]:
raw_sample = filter_df['full_text'].iloc[0]
case_folding = casefolding(raw_sample)

print('Raw data\t: ', raw_sample)
print('Case folding\t: ', case_folding)

Raw data	:  @tanyakanrl 2025: Coretax Danantara Mega korupsi(PLN Pertamina BJB) KaburAjaDulu IndonesiaGelap Grooming &amp; Pedofil CPNS Dwifungsi Abri PHK Masal MBG Efisiensi Anggaran Banjir dll Satt baru jg maret dah banyak amat masalahnya
Case folding	:  @tanyakanrl 2025: coretax danantara mega korupsi(pln pertamina bjb) kaburajadulu indonesiagelap grooming &amp; pedofil cpns dwifungsi abri phk masal mbg efisiensi anggaran banjir dll satt baru jg maret dah banyak amat masalahnya


#### 1d. Stopwords

In [31]:
# Cek jumlah stopwords berbahasa Indonesia yang disediakan NLTK
len(stopwords_ind)

758

In [32]:
# Lihat daftar stopword yang disediakan NLTK
stopwords_ind[:100]

['ada',
 'adalah',
 'adanya',
 'adapun',
 'agak',
 'agaknya',
 'agar',
 'akan',
 'akankah',
 'akhir',
 'akhiri',
 'akhirnya',
 'aku',
 'akulah',
 'amat',
 'amatlah',
 'anda',
 'andalah',
 'antar',
 'antara',
 'antaranya',
 'apa',
 'apaan',
 'apabila',
 'apakah',
 'apalagi',
 'apatah',
 'artinya',
 'asal',
 'asalkan',
 'atas',
 'atau',
 'ataukah',
 'ataupun',
 'awal',
 'awalnya',
 'bagai',
 'bagaikan',
 'bagaimana',
 'bagaimanakah',
 'bagaimanapun',
 'bagi',
 'bagian',
 'bahkan',
 'bahwa',
 'bahwasanya',
 'baik',
 'bakal',
 'bakalan',
 'balik',
 'banyak',
 'bapak',
 'baru',
 'bawah',
 'beberapa',
 'begini',
 'beginian',
 'beginikah',
 'beginilah',
 'begitu',
 'begitukah',
 'begitulah',
 'begitupun',
 'bekerja',
 'belakang',
 'belakangan',
 'belum',
 'belumlah',
 'benar',
 'benarkah',
 'benarlah',
 'berada',
 'berakhir',
 'berakhirlah',
 'berakhirnya',
 'berapa',
 'berapakah',
 'berapalah',
 'berapapun',
 'berarti',
 'berawal',
 'berbagai',
 'berdatangan',
 'beri',
 'berikan',
 'berikut'

In [33]:
# Membuat function untuk menghapus stopwords
def remove_stop_words(text):
  clean_words = []
  text = text.split()
  for word in text:
      if word not in stopwords_ind:
          clean_words.append(word)
  return " ".join(clean_words)

In [42]:
raw_sample = filter_df['full_text'].iloc[0]
case_folding = casefolding(raw_sample)
stopword_removal = remove_stop_words(case_folding)
cleantext = clean_text(raw_sample)

print('Raw data\t\t: ', raw_sample)
print('Cleaning\t: ', cleantext)
print('Case folding\t\t: ', case_folding)
print('Stopword removal\t: ', stopword_removal)

Raw data		:  @tanyakanrl 2025: Coretax Danantara Mega korupsi(PLN Pertamina BJB) KaburAjaDulu IndonesiaGelap Grooming &amp; Pedofil CPNS Dwifungsi Abri PHK Masal MBG Efisiensi Anggaran Banjir dll Satt baru jg maret dah banyak amat masalahnya
Cleaning	:  Coretax Danantara Mega korupsiPLN Pertamina BJB KaburAjaDulu IndonesiaGelap Grooming amp Pedofil CPNS Dwifungsi Abri PHK Masal MBG Efisiensi Anggaran Banjir dll Satt baru jg maret dah banyak amat masalahnya
Case folding		:  @tanyakanrl 2025: coretax danantara mega korupsi(pln pertamina bjb) kaburajadulu indonesiagelap grooming &amp; pedofil cpns dwifungsi abri phk masal mbg efisiensi anggaran banjir dll satt baru jg maret dah banyak amat masalahnya
Stopword removal	:  @tanyakanrl 2025: coretax danantara mega korupsi(pln pertamina bjb) kaburajadulu indonesiagelap grooming &amp; pedofil cpns dwifungsi abri phk masal mbg efisiensi anggaran banjir dll satt jg maret dah


#### 1e. Apply semua step filtering

Mengaplikasikan tahapan cleaning, casefolding, dan stopwords ke dalam dataset

In [35]:
# Buat function untuk mengaplikasikan seluruh step Filtering

def text_preprocessing_process(text):
  text = clean_text(text)
  text = casefolding(text)
  text = remove_stop_words(text)
  return text

In [36]:
df['cleaned_text'] = df['full_text'].apply(text_preprocessing_process)
df[['full_text', 'cleaned_text']]

,full_text,cleaned_text
0,Danantara Indonesia berkomitmen untuk mendoron...,danantara indonesia berkomitmen mendorong tran...
1,@Mdy_Asmara1701 Kalau bs efisiensi Logikanya a...,bs efisiensi logikanya anggaran surplus
2,@CNNIndonesia Apa sih buk! Noh efisiensi tapi ...,sih buk noh efisiensi pejabat asik asik
3,@Mdy_Asmara1701 Efisiensi cuma lari ke dananta...,efisiensi lari danantaro makan bergizi gratis ...
4,@geloraco Kiw kiwww hiburan baru buat dana bar...,kiw kiwww hiburan dana efisiensi diperpanjang
...,...,...
2510,@islah_bahrawi Maju PSI!!! Hidup Efisiensi!! H...,maju psi hidup efisiensi hidup jokowi mbelgedes
2511,Indonesia jam 10 lampu harus udah mati semua g...,indonesia jam lampu udah mati gara efisiensi a...
2512,Kebenaran Efektif adalah hal yang tercermin da...,kebenaran efektif tercermin tindakan sikap sik...
2513,@EricRicaldo @prabowo Presidennya Paham efisie...,presidennya paham efisiensi ga gengsi tau ah ono


### **2. Stemming ⭐**


#### 2a. Library Sastrawi untuk Stemming

In [39]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Buat fungsi untuk langkah stemming bahasa Indonesia
def stemming(text):
  text = stemmer.stem(text)
  return text

In [41]:
raw_sample = filter_df['full_text'].iloc[1]

# Step 1: Case folding
case_folding = casefolding(raw_sample)

# Step 2: Cleaning
cleantext = clean_text(case_folding)

# Step 3: Stopword removal
stopword_removal = remove_stop_words(cleantext)

# Step 4: Stemming
text_stemming = stemming(stopword_removal)

# Print hasil setiap tahap
print('Raw data\t\t: ', raw_sample)
print('Case folding\t\t: ', case_folding)
print('Cleaning\t\t: ', cleantext)
print('Stopword removal\t: ', stopword_removal)
print('Stemming\t\t: ', text_stemming)


Raw data		:  Itu bukan solusi. Solusi &amp; efek jera untuk para koruptor yang paling paten itu: miskinkan (rampas semua aset hasil korup) hukuman cukup 5 tahun di lapas umum--ini efektif dan efisiensi. Dengan catatan justice system berintegritas.
Case folding		:  itu bukan solusi. solusi &amp; efek jera untuk para koruptor yang paling paten itu: miskinkan (rampas semua aset hasil korup) hukuman cukup 5 tahun di lapas umum--ini efektif dan efisiensi. dengan catatan justice system berintegritas.
Cleaning		:  itu bukan solusi solusi amp efek jera untuk para koruptor yang paling paten itu miskinkan rampas semua aset hasil korup hukuman cukup tahun di lapas umumini efektif dan efisiensi dengan catatan justice system berintegritas
Stopword removal	:  solusi solusi amp efek jera koruptor paten miskinkan rampas aset hasil korup hukuman lapas umumini efektif efisiensi catatan justice system berintegritas
Stemming		:  solusi solusi amp efek jera koruptor paten miskin rampas aset hasil korup huk

Case folding dulu → supaya semua huruf kecil.

Cleaning teks → buang karakter aneh (emoji, tanda baca, dll).

Stopword removal → hapus kata-kata tidak penting ("yang", "dan", dll).

Stemming → baru stemming kata ke bentuk dasar.

#### 2b. Apply/terapkan stemming pada data yang sudah menerapkan proses filtering

In [43]:
# Buat function untuk mengaplikasikan stemming

def text_stemming(text):
  text = stemming(text)
  return text

In [44]:
# Tambahkan kolom 'stemmed_text' untuk melakukan stemming pada data di kolom 'cleaned_text'
df['stemmed_text'] = df['cleaned_text'].apply(text_stemming)

# Bandingkan output di kolom full_text dan cleaned_text setelah di-stemming
df[['cleaned_text', 'stemmed_text']]

,cleaned_text,stemmed_text
0,danantara indonesia berkomitmen mendorong tran...,danantara indonesia komitmen dorong transforma...
1,bs efisiensi logikanya anggaran surplus,bs efisiensi logika anggar surplus
2,sih buk noh efisiensi pejabat asik asik,sih buk noh efisiensi jabat asik asik
3,efisiensi lari danantaro makan bergizi gratis ...,efisiensi lari danantaro makan gizi gratis bik...
4,kiw kiwww hiburan dana efisiensi diperpanjang,kiw kiwww hibur dana efisiensi panjang
...,...,...
2510,maju psi hidup efisiensi hidup jokowi mbelgedes,maju psi hidup efisiensi hidup jokowi mbelgedes
2511,indonesia jam lampu udah mati gara efisiensi a...,indonesia jam lampu udah mati gara efisiensi a...
2512,kebenaran efektif tercermin tindakan sikap sik...,benar efektif cermin tindak sikap sikap angkuh...
2513,presidennya paham efisiensi ga gengsi tau ah ono,presiden paham efisiensi ga gengsi tau ah ono


bisa dilihat ada beberapa kata berimbuhan yang telah dipotong menjadi kata dasar. contoh:
1. berkomitmen >> komitmen
2. pemerintah >> perintah
3. kepastian >> pasti
4. anggaran >> anggar
5. kebijakan >> bijak
6. dst.

### **3. Tokenisasi**

In [ ]:
!pip install nltk

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
# prompt: Lakukan tokenisasi

import nltk
nltk.download('punkt')
nltk.download('stopwords')

from nltk.tokenize import word_tokenize




[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


#### 3a. Melakukan tokenisasi dari kolom stemmed_text

In [ ]:
# Buat function untuk tokenisasi text
def tokenize_text(text):
  tokens = word_tokenize(text)
  return tokens

In [ ]:
# akan ada kolom baru bernama "tokenized_text" yang berisi token hasil cleaned_text
df['tokenized_text'] = df['stemmed_text'].apply(tokenize_text)
print(df[['stemmed_text', 'tokenized_text']].head())

                                        stemmed_text  \
0  danantara indonesia komitmen dorong transforma...   
1                 bs efisiensi logika anggar surplus   
2              sih buk noh efisiensi jabat asik asik   
3  efisiensi lari danantaro makan gizi gratis bik...   
4             kiw kiwww hibur dana efisiensi panjang   

                                      tokenized_text  
0  [danantara, indonesia, komitmen, dorong, trans...  
1           [bs, efisiensi, logika, anggar, surplus]  
2      [sih, buk, noh, efisiensi, jabat, asik, asik]  
3  [efisiensi, lari, danantaro, makan, gizi, grat...  
4      [kiw, kiwww, hibur, dana, efisiensi, panjang]  


In [ ]:
df[['cleaned_text', 'stemmed_text', 'tokenized_text']]

,cleaned_text,stemmed_text,tokenized_text
0,danantara indonesia berkomitmen mendorong tran...,danantara indonesia komitmen dorong transforma...,"[danantara, indonesia, komitmen, dorong, trans..."
1,bs efisiensi logikanya anggaran surplus,bs efisiensi logika anggar surplus,"[bs, efisiensi, logika, anggar, surplus]"
2,sih buk noh efisiensi pejabat asik asik,sih buk noh efisiensi jabat asik asik,"[sih, buk, noh, efisiensi, jabat, asik, asik]"
3,efisiensi lari danantaro makan bergizi gratis ...,efisiensi lari danantaro makan gizi gratis bik...,"[efisiensi, lari, danantaro, makan, gizi, grat..."
4,kiw kiwww hiburan dana efisiensi diperpanjang,kiw kiwww hibur dana efisiensi panjang,"[kiw, kiwww, hibur, dana, efisiensi, panjang]"
...,...,...,...
2510,maju psi hidup efisiensi hidup jokowi mbelgedes,maju psi hidup efisiensi hidup jokowi mbelgedes,"[maju, psi, hidup, efisiensi, hidup, jokowi, m..."
2511,indonesia jam lampu udah mati gara efisiensi a...,indonesia jam lampu udah mati gara efisiensi a...,"[indonesia, jam, lampu, udah, mati, gara, efis..."
2512,kebenaran efektif tercermin tindakan sikap sik...,benar efektif cermin tindak sikap sikap angkuh...,"[benar, efektif, cermin, tindak, sikap, sikap,..."
2513,presidennya paham efisiensi ga gengsi tau ah ono,presiden paham efisiensi ga gengsi tau ah ono,"[presiden, paham, efisiensi, ga, gengsi, tau, ..."


#### 3b. Menghitung Frekuensi Token

In [ ]:
# Cek frekuensi (jumlah kemunculan) token

cek_freq = df['tokenized_text'].explode()  # Pisahkan semua kata/token dari setiap baris menggunakan fungsi explode()
cek_freq = cek_freq.value_counts()                # Hitung jumlah kata/token menggunakan fungsi value_counts
cek_freq.head(10)                                 # Tampilkan 10 data teratas

,count
tokenized_text,
efisiensi,2560
kena,520
anggar,492
yg,487
ga,283
ya,232
aja,216
gak,201
perintah,170


## C. Hasil Akhir Preprocessing

Pada pertemuan Preprocessing II ini, kita sudah berhasil menerapkan tahapan pembersihan data dasar di PBA mulai dari menghilangkan karakter, casefolding, remove stopwords, dan stemming

In [45]:
print(df.columns)

Index(['conversation_id_str', 'created_at', 'favorite_count', 'full_text',
       'id_str', 'image_url', 'in_reply_to_screen_name', 'lang', 'location',
       'quote_count', 'reply_count', 'retweet_count', 'tweet_url',
       'user_id_str', 'username', 'cleaned_text', 'stemmed_text'],
      dtype='object')


In [ ]:
# Lihat perbandingan tweet asli, tweet yang sudah di-preprocessing, dan token

df_fix = df[['created_at', 'favorite_count', 'in_reply_to_screen_name', 'lang', 'location',
       'quote_count', 'reply_count', 'retweet_count', 'full_text', 'cleaned_text', 'stemmed_text', 'tokenized_text']]
df_fix

,created_at,favorite_count,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,full_text,cleaned_text,stemmed_text,tokenized_text
0,Thu Mar 13 23:31:50 +0000 2025,0,NaN,in,Papua Barat,0,1,0,Danantara Indonesia berkomitmen untuk mendoron...,danantara indonesia berkomitmen mendorong tran...,danantara indonesia komitmen dorong transforma...,"[danantara, indonesia, komitmen, dorong, trans..."
1,Thu Mar 13 23:31:20 +0000 2025,0,Mdy_Asmara1701,in,NaN,0,0,0,@Mdy_Asmara1701 Kalau bs efisiensi Logikanya a...,bs efisiensi logikanya anggaran surplus,bs efisiensi logika anggar surplus,"[bs, efisiensi, logika, anggar, surplus]"
2,Thu Mar 13 23:29:40 +0000 2025,0,CNNIndonesia,in,NaN,0,0,0,@CNNIndonesia Apa sih buk! Noh efisiensi tapi ...,sih buk noh efisiensi pejabat asik asik,sih buk noh efisiensi jabat asik asik,"[sih, buk, noh, efisiensi, jabat, asik, asik]"
3,Thu Mar 13 23:27:17 +0000 2025,0,Mdy_Asmara1701,in,NaN,0,0,0,@Mdy_Asmara1701 Efisiensi cuma lari ke dananta...,efisiensi lari danantaro makan bergizi gratis ...,efisiensi lari danantaro makan gizi gratis bik...,"[efisiensi, lari, danantaro, makan, gizi, grat..."
4,Thu Mar 13 23:22:25 +0000 2025,0,geloraco,in,NaN,0,0,0,@geloraco Kiw kiwww hiburan baru buat dana bar...,kiw kiwww hiburan dana efisiensi diperpanjang,kiw kiwww hibur dana efisiensi panjang,"[kiw, kiwww, hibur, dana, efisiensi, panjang]"
...,...,...,...,...,...,...,...,...,...,...,...,...
2510,Sun Mar 09 05:21:25 +0000 2025,1,islah_bahrawi,in,Indonesia,0,0,0,@islah_bahrawi Maju PSI!!! Hidup Efisiensi!! H...,maju psi hidup efisiensi hidup jokowi mbelgedes,maju psi hidup efisiensi hidup jokowi mbelgedes,"[maju, psi, hidup, efisiensi, hidup, jokowi, m..."
2511,Sun Mar 09 05:20:53 +0000 2025,0,NaN,in,Shattering skies above,0,0,0,Indonesia jam 10 lampu harus udah mati semua g...,indonesia jam lampu udah mati gara efisiensi a...,indonesia jam lampu udah mati gara efisiensi a...,"[indonesia, jam, lampu, udah, mati, gara, efis..."
2512,Sun Mar 09 05:19:41 +0000 2025,0,NaN,in,Jahannam Lt. Underground,0,1,0,Kebenaran Efektif adalah hal yang tercermin da...,kebenaran efektif tercermin tindakan sikap sik...,benar efektif cermin tindak sikap sikap angkuh...,"[benar, efektif, cermin, tindak, sikap, sikap,..."
2513,Sun Mar 09 05:18:33 +0000 2025,0,EricRicaldo,in,NaN,0,0,0,@EricRicaldo @prabowo Presidennya Paham efisie...,presidennya paham efisiensi ga gengsi tau ah ono,presiden paham efisiensi ga gengsi tau ah ono,"[presiden, paham, efisiensi, ga, gengsi, tau, ..."


In [ ]:
df_fix.to_csv('cleaned_tweet_efisiensi.csv')

## **POSTEST**

Filter data terlebih dahulu dari rentang 140-500

In [49]:
df = pd.read_csv('tweet_efisiensi.csv')
df = df[140:501]
df

,conversation_id_str,created_at,favorite_count,full_text,id_str,image_url,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,tweet_url,user_id_str,username
140,1900208828689637603,Thu Mar 13 15:34:23 +0000 2025,0,katenye efisiensi tapi sikit² nak bikin bangunan,1900208828689637603,NaN,NaN,in,Pontianak,0,0,0,https://x.com/FathurruhtaF/status/190020882868...,99725732,FathurruhtaF
141,1900109681655505334,Thu Mar 13 15:34:02 +0000 2025,4,@tempodotco Katanya efisiensi @sjafriesjams ??...,1900208739841654845,NaN,tempodotco,in,she/her,0,0,0,https://x.com/seokjin012486/status/19002087398...,303159065,seokjin012486
142,1900109681655505334,Thu Mar 13 15:32:03 +0000 2025,0,@tempodotco Yuk efisiensi yuk..,1900208241285750993,NaN,tempodotco,in,Indonesia,0,0,0,https://x.com/ShopeeDate/status/19002082412857...,101900202,ShopeeDate
143,1900171247054708803,Thu Mar 13 15:31:04 +0000 2025,0,@xquitavee @prabowo Kelamaan tuh nunggu penjar...,1900207994929123734,NaN,xquitavee,in,Indonesia.,0,0,0,https://x.com/K7_m0en/status/1900207994929123734,3299145104,K7_m0en
144,1900167027542921512,Thu Mar 13 15:29:58 +0000 2025,0,@itsyourapon dipadetin jir fapet. curiga efisi...,1900207716645130389,NaN,itsyourapon,es,fapetugm,0,1,0,https://x.com/ineedtofurn/status/1900207716645...,1586253391365255168,ineedtofurn
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,1900044160696279050,Thu Mar 13 04:40:03 +0000 2025,0,Ceritanya bln Feb kmrn daftar pemasangan PGN (...,1900044160696279050,NaN,NaN,in,Indonesia,0,0,0,https://x.com/sidhanty/status/1900044160696279050,78840661,sidhanty
497,1900044050956419246,Thu Mar 13 04:39:37 +0000 2025,0,Kampus Negeri Mulai Terapkan Efisiensi Mahasis...,1900044050956419246,NaN,NaN,in,Jakarta,0,0,0,https://x.com/kalderanews/status/1900044050956...,817669043792420864,kalderanews
498,1900043456451686660,Thu Mar 13 04:37:15 +0000 2025,1,Pensiun tidak kena efisiensi tetap dibayarkan ...,1900043456451686660,https://pbs.twimg.com/media/Gl5PcU6XYAAGmPE.jpg,NaN,in,anywhere,0,1,0,https://x.com/ape_mezzala/status/1900043456451...,1418190228053917698,ape_mezzala
499,1900042752509042893,Thu Mar 13 04:34:27 +0000 2025,0,Belajar di gedung teknik udah kayak lagi survi...,1900042752509042893,https://pbs.twimg.com/media/Gl5OzOCbYAAZ7C4.jpg,NaN,in,"Jakarta Selatan, DKI Jakarta",3,1,0,https://x.com/noctuery/status/1900042752509042893,1364929039761203210,noctuery


2. Dari data yang telah difilter tersebut, filter lagi data yang hanya memuat kata "rakyat"

In [58]:
df = df[df['full_text'].str.contains("rakyat", case=False, na=False)]
df.head()

,conversation_id_str,created_at,favorite_count,full_text,id_str,image_url,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,tweet_url,user_id_str,username
3,1900150673057972397,Thu Mar 13 23:27:17 +0000 2025,0,@Mdy_Asmara1701 Efisiensi cuma lari ke dananta...,1900327836311052770,NaN,Mdy_Asmara1701,in,NaN,0,0,0,https://x.com/wadestahh/status/190032783631105...,1880516316920266752,wadestahh
7,1900325664370418123,Thu Mar 13 23:18:39 +0000 2025,0,hari ini dapet info kalo dana BOS dipotong 50%...,1900325664370418123,NaN,NaN,in,NaN,0,0,0,https://x.com/setelaaa/status/1900325664370418123,66318567,setelaaa
14,1900150673057972397,Thu Mar 13 23:08:35 +0000 2025,0,@Mdy_Asmara1701 Lah Kan efisiensi itu untuk da...,1900323130079666387,NaN,Mdy_Asmara1701,in,NaN,0,0,0,https://x.com/AyahBun6539919/status/1900323130...,1666289272218071040,AyahBun6539919
22,1900109681655505334,Thu Mar 13 22:44:10 +0000 2025,0,@tempodotco Efisiensi untuk kebijakan rakyat S...,1900316988205072831,NaN,tempodotco,in,Indonesia,0,0,0,https://x.com/jiminsantoso/status/190031698820...,1104516029793759232,jiminsantoso
52,1900111771035423005,Thu Mar 13 21:28:31 +0000 2025,0,@geloraco Kalo diperhatiin kayaknya pembahasan...,1900297947222573544,NaN,geloraco,in,"Tangerang, Indonesia",0,0,0,https://x.com/rezpahlevi/status/19002979472225...,177497943,rezpahlevi


Lakukan tahapan 1a-2b pada tersebut

In [59]:
df['cleaned_text'] = df['full_text'].apply(text_preprocessing_process)
df[['full_text', 'cleaned_text']]

,full_text,cleaned_text
3,@Mdy_Asmara1701 Efisiensi cuma lari ke dananta...,efisiensi lari danantaro makan bergizi gratis ...
7,hari ini dapet info kalo dana BOS dipotong 50%...,dapet info kalo dana bos dipotong krn efisiens...
14,@Mdy_Asmara1701 Lah Kan efisiensi itu untuk da...,efisiensi danantara alokasi dana kesana ya ga ...
22,@tempodotco Efisiensi untuk kebijakan rakyat S...,efisiensi kebijakan rakyat yg diatas kuwe keku...
52,@geloraco Kalo diperhatiin kayaknya pembahasan...,kalo diperhatiin kayaknya pembahasan presiden ...
59,@mas_veel Harusnya yang dikejar hutang sedikit...,dikejar hutang biar ruang fiskal negara longga...
70,Dunia sedang tidak baik SAJA termasuk kemungki...,dunia as resesi meningkat warisan fiskal buruk...
101,@creepyroom Gw malah salfok sama mobilnya yang...,gw salfok mobilnya dibiarin mangkrak bertahunt...
116,@MRahayuna @StefanAntonio__ lah cara efisiensi...,efisiensinya ga bener ngaruh aspek kehidupan n...
141,@tempodotco Katanya efisiensi @sjafriesjams ??...,efisiensi rakyat sengsara dg beragam efisiensi...


In [61]:
# Tambahkan kolom 'stemmed_text' untuk melakukan stemming pada data di kolom 'cleaned_text'
df['stemmed_text'] = df['cleaned_text'].apply(text_stemming)

# Bandingkan output di kolom full_text dan cleaned_text setelah di-stemming
df[['cleaned_text', 'stemmed_text']]

,cleaned_text,stemmed_text
3,efisiensi lari danantaro makan bergizi gratis ...,efisiensi lari danantaro makan gizi gratis bik...
7,dapet info kalo dana bos dipotong krn efisiens...,dapet info kalo dana bos potong krn efisiensi ...
14,efisiensi danantara alokasi dana kesana ya ga ...,efisiensi danantara alokasi dana kesana ya ga ...
22,efisiensi kebijakan rakyat yg diatas kuwe keku...,efisiensi bijak rakyat yg atas kuwe kuasa
52,kalo diperhatiin kayaknya pembahasan presiden ...,kalo diperhatiin kayak bahas presiden yg keluh...
59,dikejar hutang biar ruang fiskal negara longga...,kejar hutang biar ruang fiskal negara longgar ...
70,dunia as resesi meningkat warisan fiskal buruk...,dunia as resesi tingkat waris fiskal buruk pre...
101,gw salfok mobilnya dibiarin mangkrak bertahunt...,gw salfok mobil dibiarin mangkrak bertahuntahu...
116,efisiensinya ga bener ngaruh aspek kehidupan n...,efisiensi ga bener ngaruh aspek hidup noh kary...
141,efisiensi rakyat sengsara dg beragam efisiensi...,efisiensi rakyat sengsara dg agam efisiensi am...


4. Tulis 5 kata berimbuhan dan hasil stemming yang ada dalam data tersebut


- penegak = tegak
- mobilnya = mobil
- dikejar = kejar
- dipotong = potong
- berguna = guna


5. Lakukan tokenisasi pada data tersebut


In [67]:
# akan ada kolom baru bernama "tokenized_text" yang berisi token hasil cleaned_text
df['tokenized_text'] = df['stemmed_text'].apply(tokenize_text)
print(df[['stemmed_text', 'tokenized_text']].head())

                                         stemmed_text  \
3   efisiensi lari danantaro makan gizi gratis bik...   
7   dapet info kalo dana bos potong krn efisiensi ...   
14  efisiensi danantara alokasi dana kesana ya ga ...   
22          efisiensi bijak rakyat yg atas kuwe kuasa   
52  kalo diperhatiin kayak bahas presiden yg keluh...   

                                       tokenized_text  
3   [efisiensi, lari, danantaro, makan, gizi, grat...  
7   [dapet, info, kalo, dana, bos, potong, krn, ef...  
14  [efisiensi, danantara, alokasi, dana, kesana, ...  
22  [efisiensi, bijak, rakyat, yg, atas, kuwe, kuasa]  
52  [kalo, diperhatiin, kayak, bahas, presiden, yg...  


6. Tampilkan tiga token teratas yang dominan pada data tersebut


In [68]:
# Cek frekuensi (jumlah kemunculan) token

cek_freq = df['tokenized_text'].explode()  # Pisahkan semua kata/token dari setiap baris menggunakan fungsi explode()
cek_freq = cek_freq.value_counts()                # Hitung jumlah kata/token menggunakan fungsi value_counts
cek_freq.head(3)

,count
tokenized_text,
efisiensi,18
rakyat,14
yg,7
